# Task 2: SQL for Data Extraction

**Internship:** ApexPlanet Data Analytics  
**Dataset:** Cleaned Superstore Sales dataset  
**Tools:** Python, Pandas, SQLite, SQLAlchemy, Jupyter Notebook  

## Objective
Load cleaned sales data into a SQLite database and answer business questions using SQL.

In [ ]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine

project_root = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data").is_dir() and (folder / "notebooks").is_dir()
)

csv_path = project_root / "data" / "processed" / "superstore_cleaned.csv"
database_folder = project_root / "data" / "database"
database_folder.mkdir(parents=True, exist_ok=True)

database_path = database_folder / "superstore.db"

In [ ]:
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
display(df.head())

print(df.columns.tolist())

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["order_date"] = df["order_date"].dt.strftime("%Y-%m-%d")

print(df.columns.tolist())

In [ ]:
customers = (
    df[["customer_id", "customer_name", "segment"]]
    .drop_duplicates(subset="customer_id")
    .copy()
)

sales_orders = df.drop(columns=["customer_name", "segment"]).copy()

engine = create_engine(f"sqlite:///{database_path}")

customers.to_sql("customers", engine, if_exists="replace", index=False)
sales_orders.to_sql("sales_orders", engine, if_exists="replace", index=False)

print("Database created successfully.")
print("Customers:", len(customers))
print("Sales order rows:", len(sales_orders))

In [ ]:
tables_query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

display(pd.read_sql_query(tables_query, engine))

In [ ]:
display(pd.read_sql_query("SELECT * FROM customers LIMIT 5;", engine))
display(pd.read_sql_query("SELECT * FROM sales_orders LIMIT 5;", engine))

In [ ]:
query = """
SELECT *
FROM sales_orders
LIMIT 5;
"""

display(pd.read_sql_query(query, engine))

In [ ]:
query = """
SELECT product_name, sales, profit
FROM sales_orders
WHERE profit < 0
ORDER BY profit ASC
LIMIT 10;
"""

display(pd.read_sql_query(query, engine))

In [ ]:
display(pd.read_sql_query(query, engine))

# SQL Findings

1. The total sales were [your result], with a total profit of [your result].
2. The highest-selling product was [product name], generating [sales amount].
3. The highest-spending customer was [customer name], with [amount] in sales.
4. The most profitable region was [region], generating [profit amount].
5. [Sub-category] was loss-making, so its pricing and discount strategy should be reviewed.
6. Higher discounts [were / were not] associated with lower average profit.